In [1]:
from __future__ import annotations

import re
import time
from dataclasses import dataclass
from typing import Optional, Dict, List, Tuple

import pandas as pd
from tqdm import tqdm
import re
from rapidfuzz import process, fuzz

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# -----------------------------
# Helpers
# -----------------------------
SUFFIXES = {"jr", "sr", "ii", "iii", "iv", "v"}

def norm_name(name: str) -> str:
    s = re.sub(r"[^\w\s]", " ", str(name).lower())
    parts = [p for p in s.split() if p not in SUFFIXES]
    return " ".join(parts)

def classify_fbs(team_text: Optional[str]) -> Optional[str]:
    # If you have your own P4/G5 map, plug it here.
    # Otherwise leave blank and/or fill later.
    if not team_text:
        return None
    return None

def safe_text(el) -> str:
    try:
        return el.text.strip()
    except Exception:
        return ""

@dataclass
class Player247:
    name: str
    id_247: Optional[int] = None

    position: Optional[str] = None

    hs_name: Optional[str] = None
    hs_city: Optional[str] = None
    hs_state: Optional[str] = None
    hs_class: Optional[str] = None  # "Class 2021"
    hs_exp: Optional[str] = None    # "Exp 2021 - 2024" (college page)

    hs_stars: Optional[int] = None
    hs_rating_247: Optional[float] = None
    hs_natl_rank: Optional[int] = None
    hs_pos_rank: Optional[int] = None

    transfer_year: Optional[int] = None
    transfer_origin: Optional[str] = None
    transfer_destination: Optional[str] = None
    transfer_stars: Optional[int] = None
    transfer_rating: Optional[float] = None   # could be 0.9200 or 92 depending on page block
    transfer_ovr_rank: Optional[int] = None
    transfer_pos_rank: Optional[int] = None

    source_player_url: Optional[str] = None
    source_hs_url: Optional[str] = None


In [2]:
# -----------------------------
# Selenium setup
# -----------------------------
def make_driver(headless: bool = True) -> webdriver.Chrome:
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1400,1000")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    # helps reduce some bot friction
    opts.add_argument("--lang=en-US")
    driver = webdriver.Chrome(options=opts)  # Selenium Manager will fetch driver if needed
    driver.set_page_load_timeout(45)
    return driver


In [3]:
# -----------------------------
# 247 scraping primitives
# -----------------------------
def click_load_more_until_done(driver: webdriver.Chrome, timeout: int = 12, max_clicks: int = 200) -> None:
    for _ in range(max_clicks):
        try:
            btn = WebDriverWait(driver, timeout).until(
                EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Load More Players') or contains(., 'Load More')]"))
            )
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
            time.sleep(0.3)
            btn.click()
            time.sleep(0.8)  # let items render
        except Exception:
            break

def scrape_portal_player_links(driver: webdriver.Chrome, portal_url: str) -> List[str]:
    driver.get(portal_url)
    time.sleep(1.5)

    click_load_more_until_done(driver)

    # Player name links in the list are typically <a> elements; grab all /player/ URLs.
    anchors = driver.find_elements(By.XPATH, "//a[contains(@href, '/player/') and not(contains(@href, '#'))]")
    urls = []
    for a in anchors:
        href = a.get_attribute("href")
        if href and "/player/" in href:
            urls.append(href.split("?")[0].rstrip("/"))
    # dedupe while preserving order
    seen = set()
    out = []
    for u in urls:
        if u not in seen:
            out.append(u)
            seen.add(u)
    return out

In [5]:
driver = make_driver(headless=True)  # run visible for debugging
urls_2025 = scrape_portal_player_links(driver, "https://247sports.com/season/2025-football/transferportaltop/")
len(urls_2025), urls_2025[2:5]

(3010,
 ['https://247sports.com/player/nico-iamaleava-46101236/college-289779',
  'https://247sports.com/player/isaiah-world-46100635/college-325778',
  'https://247sports.com/player/damon-wilson-ii-46114588/college-291463'])

In [6]:
#### add tqdm to this

In [7]:
def get_text(driver, xpath, timeout=10):
    el = WebDriverWait(driver, timeout).until(EC.presence_of_element_located((By.XPATH, xpath)))
    return el.text.strip()

def body_text(driver):
    return driver.find_element(By.TAG_NAME, "body").text

def extract_player_id_247(url: str):
    m = re.search(r"/player/[^/]+-(\d+)", url)
    return int(m.group(1)) if m else None

def parse_int_from_text(s: str) -> Optional[int]:
    m = re.search(r"(\d{1,6})", s.replace(",", ""))
    return int(m.group(1)) if m else None

def parse_float_from_text(s: str) -> Optional[float]:
    m = re.search(r"(\d+\.\d+|\d+)", s.replace(",", ""))
    return float(m.group(1)) if m else None

def scrape_player_pages(driver: webdriver.Chrome, player_url: str) -> Player247:
    driver.get(player_url)
    time.sleep(1.2)

    p = Player247(name="", source_player_url=player_url)

    m = re.search(r"/player/[^/]+-(\d+)", player_url)
    if m:
        p.id_247 = int(m.group(1)) if m else None


    # Name (h1)
    try:
        p.name = safe_text(driver.find_element(By.XPATH, "//h1"))
    except Exception:
        p.name = player_url

    # Position (look for "Pos XX")
    try:
        pos_line = driver.find_element(By.XPATH, "//*[contains(., 'Pos') and contains(., 'Height')]")
        # fallback if this fails; below handles main profile block
    except Exception:
        pass

    # More reliable: find the "Pos" label line
    try:
        pos_el = driver.find_element(By.XPATH, "//*[contains(., 'Pos') and contains(., 'Pos')]/following::*[1]")
    except Exception:
        pos_el = None

    # Simple: search visible text fragments
    body = driver.find_element(By.TAG_NAME, "body").text

    m_pos = re.search(r"\bPos\s+([A-Z]{1,4})\b", body)
    if m_pos:
        p.position = m_pos.group(1).strip()

    # Prospect Info: High School / City / Exp
    m_hs = re.search(r"High School\s+([^\n]+)", body)
    if m_hs:
        p.hs_name = m_hs.group(1).strip()

    m_city = re.search(r"City\s+([^\n,]+),\s*([A-Z]{2})", body)
    if m_city:
        p.hs_city = m_city.group(1).strip()
        p.hs_state = m_city.group(2).strip()

    m_exp = re.search(r"\bExp\s+(\d{4}\s*-\s*\d{4})\b", body)
    if m_exp:
        p.hs_exp = f"Exp {m_exp.group(1).replace(' ', '')}"

    # Transfer rankings block: "As a Transfer" -> 247Sports Transfer Rankings
    # Example: shows "247Sports Transfer Rankings" then rating + OVR # + position #. :contentReference[oaicite:9]{index=9}
    if "247Sports Transfer Rankings" in body:
        # grab rating number that appears shortly after that phrase
        after = body.split("247Sports Transfer Rankings", 1)[1]
        lines = [ln.strip() for ln in after.splitlines() if ln.strip()]
        # Often first numeric line is rating (e.g., 92)
        if lines:
            p.transfer_rating = parse_float_from_text(lines[0])
        # Try to grab OVR rank and position rank lines:
        # "OVR 48" then "WR 8"
        ovr_match = re.search(r"\bOVR\s+(\d+)\b", after)
        posrank_match = re.search(rf"\b{re.escape(p.position or '')}\s+(\d+)\b", after) if p.position else None
        if ovr_match:
            p.transfer_ovr_rank = int(ovr_match.group(1))
        if posrank_match:
            p.transfer_pos_rank = int(posrank_match.group(1))

    # Find HS recruiting profile link ("View recruiting profile")
    hs_url = None
    try:
        hs_link = driver.find_element(By.XPATH, "//a[contains(., 'View recruiting profile')]")
        hs_url = hs_link.get_attribute("href")
    except Exception:
        # fallback: any /high-school- URL in anchors
        for a in driver.find_elements(By.XPATH, "//a[contains(@href, '/high-school-')]"):
            hs_url = a.get_attribute("href")
            break

    if hs_url:
        p.source_hs_url = hs_url.split("?")[0].rstrip("/")
        hs = scrape_hs_page(driver, p.source_hs_url)
        # merge HS fields
        p.hs_class = hs.hs_class
        p.hs_rating_247 = hs.hs_rating_247
        p.hs_natl_rank = hs.hs_natl_rank
        p.hs_pos_rank = hs.hs_pos_rank
        p.hs_stars = hs.hs_stars

    return p

def extract_transfer_block(txt: str, pos: str | None):
    if "247SPORTS TRANSFER RANKINGS" not in txt:
        return None, None, None, None
    m = re.search(r"247SPORTS TRANSFER RANKINGS\s+(\d+)\s+\((\d{4})\)", txt)
    rating = int(m.group(1)) if m else None
    year = int(m.group(2)) if m else None
    ovr = int(re.search(r"\bOVR\s+(\d+)\b", txt).group(1)) if re.search(r"\bOVR\s+(\d+)\b", txt) else None

    # pos rank: restrict search to after transfer header
    after = txt.split("247SPORTS TRANSFER RANKINGS", 1)[1]
    pos_rank = None
    if pos:
        m2 = re.search(rf"\b{re.escape(pos)}\s+(\d+)\b", after)
        pos_rank = int(m2.group(1)) if m2 else None
    return rating, year, ovr, pos_rank

def scrape_hs_recruiting_page(driver, hs_url: str):
    driver.get(hs_url)
    WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.XPATH, "//h1")))
    txt = body_text(driver)

    # Class YYYY (THIS is what goes to your Excel column H)
    hs_class = None
    m_class = re.search(r"\bClass\s+(\d{4})\b", txt)
    if m_class:
        hs_class = int(m_class.group(1))

    # 247 rating (often shows right after "247Sports" as an integer)
    hs_rating_247 = None
    m_rating = re.search(r"\b247Sports\b\s+(\d{2,3})\b", txt)
    if m_rating:
        hs_rating_247 = int(m_rating.group(1))

    # National rank (from "Natl. 2")
    hs_natl_rank = None
    m_natl = re.search(r"\bNatl\.\s+(\d{1,6})\b", txt)
    if m_natl:
        hs_natl_rank = int(m_natl.group(1))

    # Position + position rank (e.g., "QB 2")
    hs_pos = None
    hs_pos_rank = None
    m_posrank = re.search(r"\b([A-Z]{1,4})\s+(\d{1,6})\b", txt)
    # NOTE: this finds *first* token match; we want the one near Natl.
    # Better: search after Natl. line if present
    if "Natl." in txt:
        after = txt.split("Natl.", 1)[1]
        m_posrank2 = re.search(r"\b([A-Z]{1,4})\s+(\d{1,6})\b", after)
        if m_posrank2:
            hs_pos = m_posrank2.group(1)
            hs_pos_rank = int(m_posrank2.group(2))

    # Stars: often icons; fallback to None here. We'll add DOM-counting once you confirm selector.
    hs_stars = None

    return {
        "hs_class": hs_class,
        "hs_rating_247": hs_rating_247,
        "hs_natl_rank": hs_natl_rank,
        "hs_pos": hs_pos,
        "hs_pos_rank": hs_pos_rank,
        "hs_stars": hs_stars,
        "source_hs_url": hs_url
    }


def scrape_hs_page(driver: webdriver.Chrome, hs_url: str) -> Player247:
    driver.get(hs_url)
    time.sleep(1.0)
    body = driver.find_element(By.TAG_NAME, "body").text

    p = Player247(name="", source_hs_url=hs_url)

    # Name
    try:
        p.name = safe_text(driver.find_element(By.XPATH, "//h1"))
    except Exception:
        p.name = hs_url

    # Class year (e.g., "Class 2021") :contentReference[oaicite:10]{index=10}
    m_class = re.search(r"\bClass\s+(\d{4})\b", body)
    if m_class:
        p.hs_class = f"Class {m_class.group(1)}"

    # 247 rating integer shown under "247Sports" (e.g., 88) :contentReference[oaicite:11]{index=11}
    # crude but effective: find the section
    if "### 247Sports" in body or "247Sports\n\n" in body:
        # often the first standalone number after "247Sports" is the rating
        m_rating = re.search(r"247Sports\s+(\d{2})\b", body)
        if m_rating:
            p.hs_rating_247 = float(m_rating.group(1))

    # Composite Natl and position ranks :contentReference[oaicite:12]{index=12}
    m_natl = re.search(r"Natl\.\s+(\d{1,5})", body)
    if m_natl:
        p.hs_natl_rank = int(m_natl.group(1))

    # Position rank: take the first "WR 121" style line after composite
    m_pos = re.search(r"247Sports Composite®.*?\n.*?\n.*?\n\s*\*\s*([A-Z]{1,4})\s+(\d{1,5})", body, re.S)
    if m_pos:
        p.position = m_pos.group(1)
        p.hs_pos_rank = int(m_pos.group(2))

    # Stars: DOM-based. Count star icons if present.
    # You will likely need to tweak selector once you inspect the page with DevTools.
    star_count = None
    for sel in [
        "span.rankings-star",
        ".rankings-stars svg",
        ".stars svg",
        ".composite-stars svg",
    ]:
        els = driver.find_elements(By.CSS_SELECTOR, sel)
        if els and len(els) <= 7:
            star_count = len(els)
            break
    if star_count:
        p.hs_stars = star_count

    return p

def scrape_player(driver, player_url: str):
    driver.get(player_url)
    WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.XPATH, "//h1")))
    txt = body_text(driver)

    name = get_text(driver, "//h1", timeout=10)
    pid = extract_player_id_247(player_url)

    # Position from "POS QB"
    pos = None
    m_pos = re.search(r"\bPOS\s+([A-Z]{1,4})\b", txt)
    if m_pos:
        pos = m_pos.group(1)

    # Prospect info (HS, City)
    hs_name = re.search(r"HIGH SCHOOL\s+([^\n]+)", txt).group(1).strip() if "HIGH SCHOOL" in txt else None
    city = re.search(r"CITY\s+([^\n]+)", txt).group(1).strip() if "CITY" in txt else None
    hs_city, hs_state = None, None
    if city and "," in city:
        hs_city = city.split(",")[0].strip()
        hs_state = city.split(",")[1].strip()

    # Transfer block
    t_rating, t_year, t_ovr, t_posrank = extract_transfer_block(txt, pos)

    # HS recruiting URL (prefer the explicit link text)
    hs_url = None
    try:
        hs_url = driver.find_element(By.XPATH, "//a[contains(., 'View recruiting profile')]").get_attribute("href")
    except Exception:
        # fallback
        anchors = driver.find_elements(By.XPATH, "//a[contains(@href, '/high-school-')]")
        if anchors:
            hs_url = anchors[0].get_attribute("href")

    hs_data = {}
    if hs_url:
        hs_url = hs_url.split("?")[0].rstrip("/")
        hs_data = scrape_hs_recruiting_page(driver, hs_url)

    return {
        "id_247": pid,
        "name": name,
        "pos_247": pos,
        "hs_name": hs_name,
        "hs_city": hs_city,
        "hs_state": hs_state,
        "transfer_rating": t_rating,
        "transfer_year": t_year,
        "transfer_ovr_rank": t_ovr,
        "transfer_pos_rank": t_posrank,
        **hs_data,
        "source_player_url": player_url
    }


In [ ]:
p = scrape_player(driver, urls_2025[2])
p

In [9]:
scraped_df = pd.DataFrame(rows)
scraped_df

,id_247,name,pos_247,hs_name,hs_city,hs_state,transfer_rating,transfer_year,transfer_ovr_rank,transfer_pos_rank,hs_class,hs_rating_247,hs_natl_rank,hs_pos,hs_pos_rank,hs_stars,source_hs_url,source_player_url
0,46101236,Nico Iamaleava,QB,Warren,Downey,CA,98,2025,1,1.0,None,None,None,None,None,None,https://247sports.com/player/nico-iamaleava-46...,https://247sports.com/player/nico-iamaleava-46...
1,46100635,Isaiah World,OT,Lincoln,San Diego,CA,98,2025,2,1.0,None,None,None,None,None,None,https://247sports.com/player/isaiah-world-4610...,https://247sports.com/player/isaiah-world-4610...
2,46114588,Damon Wilson II,None,Venice,Venice,FL,98,2025,3,NaN,None,None,None,None,None,None,https://247sports.com/player/damon-wilson-ii-4...,https://247sports.com/player/damon-wilson-ii-4...
3,46053141,Carson Beck,QB,Mandarin,Jacksonville,FL,96,2025,4,2.0,None,None,None,None,None,None,https://247sports.com/player/carson-beck-46053...,https://247sports.com/player/carson-beck-46053...
4,46134398,Eric Singleton Jr.,WR,Alexander,Douglasville,GA,96,2025,5,1.0,None,None,None,None,None,None,https://247sports.com/player/eric-singleton-jr...,https://247sports.com/player/eric-singleton-jr...


In [8]:
# p = scrape_player(driver, urls_2025[2])
# p

# scrape 10 players
rows = []
for u in urls_2025[2:12]:
    try:
        rows.append(scrape_player(driver, u))
    except Exception as e:
        print("FAIL", u, e)

import pandas as pd
scraped_df = pd.DataFrame(rows)
scraped_df[["id_247","name","pos_247","hs_class","hs_rating_247",
            "hs_natl_rank","hs_pos_rank","transfer_rating","transfer_year",
            "transfer_ovr_rank","transfer_pos_rank"]]

FAIL https://247sports.com/player/john-mateer-46078807/college-282972 HTTPConnectionPool(host='localhost', port=58093): Read timed out. (read timeout=120)
FAIL https://247sports.com/player/patrick-payton-46084313/college-266654 HTTPConnectionPool(host='localhost', port=58093): Read timed out. (read timeout=120)
FAIL https://247sports.com/player/duce-robinson-46086662/college-298925 HTTPConnectionPool(host='localhost', port=58093): Read timed out. (read timeout=120)


KeyboardInterrupt: 

In [9]:
urls_2025[2]

'https://247sports.com/player/nico-iamaleava-46101236/college-289779'

In [7]:
driver.get(urls_2025[2])
print(driver.title)
print(driver.current_url)
print(driver.find_element(By.TAG_NAME, "body").text[:2000])

Nico Iamaleava, Tennessee, Quarterback
https://247sports.com/player/nico-iamaleava-46101236/college-289779/
TEAMS
CBS SPORTS
CBS SPORTS HQ
SPORTSLINE
MAXPREPS
SHOP
STUBHUB
BETTING
WATCH
LOG IN
JOIN
FB REC
BK REC
TRANSFER PORTAL
NCAA FB
NCAA BK
COMMUNITY
JOIN
UPGRADE & SAVE! 50% off annual VIP membership
UCLA BRUINS
ENROLLED
Photos
Nico Iamaleava
NCAA
POS
QB
HEIGHT
6-6
WEIGHT
215
Timeline
Embed
Prospect Info
HIGH SCHOOL
Warren
CITY
Downey, CA
EXP
2023 - 2025
Watch Highlights
Transfer Portal Crystal Ball®
No Crystal Ball predictions at this time.
Tennessee Volunteers
News Feed
Boards
Commits
Roster
As a Transfer
247SPORTS TRANSFER RANKINGS
98 (2025)
OVR
1
QB
1
As a Prospect
247SPORTS
100
NATL.
2
.st0{fill-rule:evenodd;clip-rule:evenodd;fill:#004B82;}
QB
2
CA
1
View recruiting profile
Playlist
1
When Work Works Presented By UKG: Nico Iamaleava
2
Penn State-UCLA: Player Of The Game Presented By Belfor
3
Is Penn State's Defense To Blame For The Wild Loss To UCLA?
4
Gary Danielson Calls UCLA

In [8]:
txt = driver.find_element(By.TAG_NAME, "body").text

import re

pos = re.search(r"\bPOS\s+([A-Z]{1,4})\b", txt).group(1)
hs = re.search(r"HIGH SCHOOL\s+([^\n]+)", txt).group(1).strip()
city = re.search(r"CITY\s+([^\n]+)", txt).group(1).strip()
exp = re.search(r"EXP\s+([0-9]{4}\s*-\s*[0-9]{4})", txt).group(1).strip()

transfer_rating = re.search(r"247SPORTS TRANSFER RANKINGS\s+(\d+)\s+\((\d{4})\)", txt).groups()
ovr = re.search(r"\bOVR\s+(\d+)\b", txt).group(1)

# position rank inside transfer section (first occurrence after transfer header)
transfer_section = txt.split("247SPORTS TRANSFER RANKINGS", 1)[1]
pos_rank = re.search(rf"\b{pos}\s+(\d+)\b", transfer_section).group(1)

pos, hs, city, exp, transfer_rating, ovr, pos_rank


('QB', 'Warren', 'Downey, CA', '2023 - 2025', ('98', '2025'), '1', '1')

#### change 'Exp' to high school class using high school url


In [ ]:
sample = []
for u in urls_2025[:15]:
    try:
        sample.append(scrape_player_pages(driver, u))
    except Exception as e:
        print("fail", u, e)

pd.DataFrame([s.__dict__ for s in sample]).head()

In [ ]:
df = pd.read_excel("data/2024-2025 Player Database v2.xlsx")
df.columns

In [ ]:
by_id, by_namepos = build_lookup(sample)
df2 = df.copy()

mask = df2["pff_snaps"].fillna(0) >= 100
for idx in df2.index[mask][:50]:  # only first 50 for test
    row = df2.loc[idx]
    p = match_player(row.get("player_id"), row.get("full_name",""), str(row.get("position","")).upper(), by_id, by_namepos)
    if not p:
        continue
    # set df2.at[idx, ...] = ...

df2.loc[mask, ["full_name","position","pff_snaps","247/On3 Position","High School, City, State"]].head(20)

In [ ]:
driver.quit()

In [ ]:
# -----------------------------
# Excel fill logic
# -----------------------------
def build_lookup(players):
    by_id = {p.id_247: p for p in players if p.id_247}
    by_namepos = {(norm_name(p.name), (p.position or "").upper()): p for p in players}
    return by_id, by_namepos

def match_player(row_player_id, row_name, row_pos, by_id, by_namepos):
    if pd.notna(row_player_id):
        pid = int(row_player_id)
        if pid in by_id:
            return by_id[pid]
    key = (norm_name(row_name), (row_pos or "").upper())
    if key in by_namepos:
        return by_namepos[key]
    # optional fuzzy fallback here...
    return None

    # fuzzy within same position
    candidates = [(k, v) for k, v in lookup.items() if k[1] == (row_pos or "").upper()]
    if not candidates:
        return None
    names = [k[0] for k, _ in candidates]
    best = process.extractOne(norm_name(row_name), names, scorer=fuzz.WRatio)
    if not best or best[1] < 92:
        return None
    matched_norm = best[0]
    for (k_norm, k_pos), v in lookup.items():
        if k_norm == matched_norm and k_pos == (row_pos or "").upper():
            return v
    return None

def fill_excel(in_path: str, out_path: str, scraped: List[Player247]) -> None:
    df = pd.read_excel(in_path)

    # Only fill for 100+ snaps
    target = df["pff_snaps"].fillna(0) >= 100
    by_id, by_namepos = build_lookup(scraped)
    row_pid = row.get("player_id")
    p = match_player(row_pid, row_name, row_pos, by_id, by_namepos)

    # columns
    F = "247/On3 Position"
    G = "High School, City, State"
    H = "247 'Exp' (aka High School Class)"
    I = "H/S Stars"
    J = "H/S Rating"
    K = "H/S National Rank"
    L = "H/S Position Rank"

    M = "Transfer Year"
    N = "Transfer Origin"
    O = "Origin P4 / G5 / Non-FBS"
    P = "Transfer Destination"
    Q = "Destination P4 / G5 / Non-FBS"
    R = "Transfer Stars"
    S = "Transfer Rating"
    T = "Transfer Overall Rank"
    U = "Transfer Position Rank"

    for idx in tqdm(df.index[target], desc="Filling rows"):
        row = df.loc[idx]
        row_name = row.get("full_name", "")
        row_pos = str(row.get("position", "")).upper()  # your sheet has 'position'
        p = match_player(row_name, row_pos, lookup)
        if not p:
            continue

        df.at[idx, F] = p.position
        if p.hs_name and p.hs_city and p.hs_state:
            df.at[idx, G] = f"{p.hs_name}, {p.hs_city}, {p.hs_state}"
        df.at[idx, H] = p.hs_exp or p.hs_class
        df.at[idx, I] = p.hs_stars
        df.at[idx, J] = p.hs_rating_247
        df.at[idx, K] = p.hs_natl_rank
        df.at[idx, L] = p.hs_pos_rank

        # Transfer: only fill if you have transfer info (else leave blank M:U)
        if p.transfer_rating or p.transfer_ovr_rank or p.transfer_pos_rank:
            df.at[idx, M] = p.transfer_year
            df.at[idx, N] = p.transfer_origin
            df.at[idx, O] = classify_fbs(p.transfer_origin)
            df.at[idx, P] = p.transfer_destination
            df.at[idx, Q] = classify_fbs(p.transfer_destination)
            df.at[idx, R] = p.transfer_stars
            df.at[idx, S] = p.transfer_rating
            df.at[idx, T] = p.transfer_ovr_rank
            df.at[idx, U] = p.transfer_pos_rank

    df.to_excel(out_path, index=False)


def main():
    portal_urls = [
        "https://247sports.com/season/2024-football/transferportaltop/",
        "https://247sports.com/season/2025-football/transferportaltop/",
    ]

    driver = make_driver(headless=True)
    try:
        all_player_urls = []
        for url in portal_urls:
            links = scrape_portal_player_links(driver, url)
            all_player_urls.extend(links)

        # Deduplicate
        all_player_urls = list(dict.fromkeys(all_player_urls))

        scraped: List[Player247] = []
        for u in tqdm(all_player_urls, desc="Scraping player pages"):
            try:
                scraped.append(scrape_player_pages(driver, u))
            except Exception:
                continue

        fill_excel(
            in_path="2024-2025 Player Database v2.xlsx",
            out_path="2024-2025 Player Database v2_FILLED.xlsx",
            scraped=scraped,
        )
    finally:
        driver.quit()

if __name__ == "__main__":
    main()